### Procesamiento de Lenguaje Natural I
# **Desafío 1**



In [1]:
%pip install numpy scikit-learn

Note: you may need to restart the kernel to use updated packages.


### Vectorización de texto y modelo de clasificación Naïve Bayes con el dataset 20 newsgroups

In [2]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score

Utilizamos **20newsgroups** por ser un dataset clásico de NLP ya viene incluido y formateado en sklearn

In [3]:
from sklearn.datasets import fetch_20newsgroups
import numpy as np

## Carga de datos

Cargamos los datos (ya separados de forma predeterminada en train y test)

El dataset 20 Newsgroups contiene aproximadamente 18 000 publicaciones de grupos de noticias distribuidas en 20 temas. Está dividido en dos subconjuntos: uno para entrenamiento (train set) y otro para pruebas (test set).

In [4]:
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

## Vectorización

Instanciamos un vectorizador.

Podemos ver diferentes parámetros de instanciación en la documentación de sklearn https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

In [5]:
tfidfvect = TfidfVectorizer()

En el atributo `data` accedemos al texto

In [6]:
print(newsgroups_train.data[0])

I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.


Con la interfaz habitual de sklearn podemos ajustar el vectorizador (obtener el vocabulario y calcular el vector IDF) y transformar directamente los datos.

Podemos denominar `X_train` como la matriz documento-término.

In [7]:
X_train = tfidfvect.fit_transform(newsgroups_train.data)

Recordemos que las vectorizaciones por conteos son de tipo sparse, por ello sklearn convenientemente devuelve los vectores de documentos como matrices de tipo sparse.

In [8]:
print(type(X_train))
print(f'shape: {X_train.shape}')
print(f'Cantidad de documentos: {X_train.shape[0]}')
print(f'Tamaño del vocabulario (dimensionalidad de los vectores): {X_train.shape[1]}')

<class 'scipy.sparse._csr.csr_matrix'>
shape: (11314, 101631)
Cantidad de documentos: 11314
Tamaño del vocabulario (dimensionalidad de los vectores): 101631


Una vez ajustado el vectorizador, podemos acceder a atributos como el vocabulario aprendido. Es un diccionario que va de términos a índices.

El índice es la posición en el vector de documento.

In [9]:
tfidfvect.vocabulary_['car']

25775

Probamos con una palbra que no está en el documento.

In [10]:
# tfidfvect.vocabulary_['cocoliso']

Es muy útil tener el diccionario opuesto que va de índices a términos

In [11]:
idx2word = {v: k for k,v in tfidfvect.vocabulary_.items()}

En `y_train` guardamos los targets que son enteros

In [12]:
y_train = newsgroups_train.target
y_train[:10]

array([ 7,  4,  4,  1, 14, 16, 13,  3,  2,  4])

Hay 20 clases correspondientes a los 20 grupos de noticias

In [13]:
print(f'clases {np.unique(newsgroups_test.target)}')
newsgroups_test.target_names

clases [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]


['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']

## Similaridad de documentos

Veamos similaridad de documentos. Tomemos algún documento

In [14]:
idx = 4811
print(newsgroups_train.data[idx])

THE WHITE HOUSE

                  Office of the Press Secretary
                   (Pittsburgh, Pennslyvania)
______________________________________________________________
For Immediate Release                         April 17, 1993     

             
                  RADIO ADDRESS TO THE NATION 
                        BY THE PRESIDENT
             
                Pittsburgh International Airport
                    Pittsburgh, Pennsylvania
             
             
10:06 A.M. EDT
             
             
             THE PRESIDENT:  Good morning.  My voice is coming to
you this morning through the facilities of the oldest radio
station in America, KDKA in Pittsburgh.  I'm visiting the city to
meet personally with citizens here to discuss my plans for jobs,
health care and the economy.  But I wanted first to do my weekly
broadcast with the American people. 
             
             I'm told this station first broadcast in 1920 when
it reported that year's presidential elec

Medimos la similaridad coseno con todos los documentos de train

In [15]:
cossim = cosine_similarity(X_train[idx], X_train)[0]

Podemos ver los valores de similaridad ordenados de mayor a menor

In [16]:
np.sort(cossim)[::-1]

array([1.        , 0.70930477, 0.67474953, ..., 0.        , 0.        ,
       0.        ], shape=(11314,))

Después vemos a qué documentos corresponden

In [17]:
np.argsort(cossim)[::-1]

array([ 4811,  6635,  4253, ...,  1534, 10055,  4750], shape=(11314,))

Obtenemos los 5 documentos más similares:

In [18]:
mostsim = np.argsort(cossim)[::-1][1:6]
print(mostsim)

[6635 4253 3596 4271 3746]


El documento original pertenece a la clase:

In [19]:
newsgroups_train.target_names[y_train[idx]]

'talk.politics.misc'

Revisamos las clases de los 5 más similares:

In [20]:
for i in mostsim:
  print(newsgroups_train.target_names[y_train[i]])

talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc


### Modelo de clasificación Naïve Bayes

Instanciamos el modelo de clasificación Naive Bayes y lo entrenamos con sklearn

In [21]:
clf = MultinomialNB()
clf.fit(X_train, y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


Ya tenemos nuestro vectorizador ya ajustado en train, vectorizamos los textos
del conjunto de test.

In [22]:
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target
y_pred =  clf.predict(X_test)

El F1-score es una métrica adecuada para evaluar el desempeño de modelos de clasificación, especialmente cuando existe desbalance entre clases.

* El promediado macro calcula el promedio del F1-score de cada clase, otorgando el mismo peso a todas las clases.
* El promediado micro calcula las métricas de forma global considerando todas las predicciones; en problemas de clasificación multiclase suele ser equivalente a la accuracy, por lo que no es la mejor métrica cuando el dataset está desbalanceado.

In [23]:
f1_score(y_test, y_pred, average='macro')

0.5854345727938506

---

## **Consigna del Desafío 1**
**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado.**



**1. Vectorizar documentos**
* Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
la similaridad según el contenido del texto y la etiqueta de clasificación.

**2. Construir un modelo de clasificación por prototipos (tipo zero-shot).**
* Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.

**3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación**

* F1-Score Macro en el conjunto de datos de test. Considerar cambiar parámetros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial y ComplementNB.

**NO cambiar el hiperparámetro ngram_range de los vectorizadores**.

**4. Transponer la matriz documento-término.**
* De esa manera se obtiene una matriz término-documento que puede ser interpretada como una colección de vectorización de palabras.
* Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares.

**Elegir las palabras MANUALMENTE para evitar la aparición de términos poco interpretables**.


---
# Resolución

Uso las variables que arma la plantilla: `newsgroups_train`, `newsgroups_test`, el vectorizador `tfidfvect` (TF-IDF sin parámetros), `X_train`, `y_train`, `X_test`, `y_test` y el baseline `clf` (`MultinomialNB()`) con sus predicciones `y_pred`. Las semillas están fijas, así que los resultados se reproducen.

In [24]:
import pandas as pd
from sklearn.base import clone
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import StratifiedKFold

target_names = newsgroups_train.target_names
vocab = tfidfvect.get_feature_names_out()

## 1. Similitud entre documentos

Tomo 5 documentos al azar (semilla 42) y busco los 5 más parecidos de cada uno por similitud coseno, sobre la matriz TF-IDF de la plantilla.

Para entender **por qué** sale cada vecino, muestro los términos que más aportan a la similitud. Como cada fila de la matriz tiene norma 1, el coseno entre dos documentos es la suma de los productos término a término, y los términos con mayor producto son los que explican el parecido.

In [25]:
def recorte(i, n=200):
    """Primeros n caracteres del documento i del train, en una sola línea."""
    return ' '.join(newsgroups_train.data[i].split())[:n]


def palabras(i):
    """Cantidad de palabras del documento i del train."""
    return len(newsgroups_train.data[i].split())


def vecinos(i, X=X_train, k=5):
    """Los k documentos más parecidos al documento i (sin contarse a sí mismo) y sus similitudes."""
    sim = cosine_similarity(X[i], X).ravel()
    sim[i] = -1
    top = np.argsort(sim)[::-1][:k]
    return top, sim[top]


def terminos_compartidos(i, j, k=5):
    """Los k términos que más aportan a la similitud coseno entre los documentos i y j."""
    aporte = X_train[i].multiply(X_train[j]).toarray().ravel()
    return [vocab[t] for t in np.argsort(aporte)[::-1][:k] if aporte[t] > 0]


rng = np.random.default_rng(42)
docs_elegidos = rng.choice(X_train.shape[0], size=5, replace=False)

for i in docs_elegidos:
    print(f'=== Documento {i} · {target_names[y_train[i]]} · {palabras(i)} palabras')
    print(recorte(i, 400), '\n')
    top, sims = vecinos(i)
    for j, s in zip(top, sims):
        marca = '✔' if y_train[j] == y_train[i] else '✘'
        print(f'  {marca} {j:>5} · {target_names[y_train[j]]:<24} · sim {s:.3f} · {palabras(j)} palabras'
              f' · comparten: {", ".join(terminos_compartidos(i, j))}')
        print(f'      {recorte(j, 150)}')
    print()

=== Documento 8754 · talk.religion.misc · 565 palabras
/(hudson) /If someone inflicts pain on themselves, whether they enjoy it or not, they /are hurting themselves. They may be permanently damaging their body. That is true. It is also none of your business. Some people may also reason that by reading the bible and being a Xtian you are permanently damaging your brain. By your logic, it would be OK for them to come into your home, take away your bible 

  ✔  6552 · talk.religion.misc       · sim 0.490 · 944 palabras · comparten: you, to, the, that, your
      If I have a habit that I really want to break, and I am willing to make whatever sacrifice I need to make to break it, then I do so. There have been b
  ✔ 10613 · talk.religion.misc       · sim 0.481 · 657 palabras · comparten: hudson, to, the, you, that
      /(hudson) /Yes you do. Who is to say that it is immoral for onesself to experience /pain or to be hurt in some other way. Maybe unpleasant, but that d
  ✔  3616 · talk.relig

**Análisis**

**8754 · `talk.religion.misc` — 4/5 de la misma clase.** Es una discusión larga, en segunda persona, sobre si hacerse daño a uno mismo es asunto de los demás (con la Biblia de por medio). Los vecinos comparten sobre todo stopwords (*you*, *to*, *the*, *that*, *your*) y el nombre *hudson*: el 10613 es otro mensaje del mismo hilo, que cita a la misma persona. El único error (8726, `talk.politics.mideast`) es otro texto largo y argumentativo en segunda persona: se parece en el estilo, no en el tema. Las similitudes son altas (0.46–0.49) porque son textos largos con muchas palabras comunes.

**4965 · `comp.sys.mac.hardware` — 2/5.** Es una sola línea: *"Plug the printer in the printer port, and the modem in the modem port"*. Los cinco vecinos hablan de puertos de impresora y módems, así que la similitud **tiene sentido por el contenido**, aunque la etiqueta no coincida: dos son de Mac, dos de PC (`comp.sys.ibm.pc.hardware`) y uno de `comp.graphics`. Un texto tan corto no tiene ninguna palabra que diga "Mac", y conectar una impresora es un tema compartido entre grupos.

**7404 · `comp.os.ms-windows.misc` — 0/5.** Pregunta cómo minimizar el *Program Manager* de Windows al abrir una aplicación. Falla por dos limitaciones de la bolsa de palabras:
- **No hay orden ni n-gramas:** *program manager* se parte en *program* y *manager*, y *manager* junto con *application* lo acercan a mensajes de X Window (`comp.windows.x`), donde se habla del *window manager*.
- **Documentos casi vacíos:** dos vecinos (8719 y 2429) son mensajes que solo dicen *"Hello,"*. Comparten una única palabra con la consulta y aun así tienen similitud 0.225, más que muchos textos con contenido real.

**1009 · `talk.politics.guns` — 4/5.** Es un mensaje "meta" sobre una discusión del grupo (el derecho a tener armas sin importar la religión). Acierta, pero con similitudes muy bajas (≈ 0.14): los vecinos comparten *guns*, *firearms*, *own* y, sobre todo, el nombre del grupo escrito en el texto (*talk*, *politics*). El error (5084, `alt.atheism`) aparece por *religion*, *history* y *regardless*.

**4899 · `sci.crypt` — 3/5.** Es un comentario político (Chomsky, *Manufacturing Consent*, el control de las redes por parte de los medios) sin vocabulario de criptografía. Los vecinos comparten solo stopwords (*the*, *of*, *it*, *be*, *will*) y son textos muy largos (el 913 tiene más de 8.600 palabras) de `sci.crypt`, `talk.politics.mideast` y `alt.atheism`. Los tres aciertos son casualidad: en este dataset `sci.crypt` tiene muchos mensajes políticos largos sobre el chip Clipper (se ve en el punto 4), llenos de stopwords.

**En resumen:** 13 de los 25 vecinos (52 %) son de la misma clase. La similitud funciona cuando el documento tiene vocabulario propio del tema (impresoras, armas, religión) y falla cuando:
- el documento es muy corto o casi vacío,
- el sentido depende del orden de las palabras (*program manager*),
- el tema se comparte entre grupos (puertos en Mac y en PC),
- el parecido lo explican las stopwords o el estilo, y no el tema.

También aparece un efecto de hilo: los mensajes que citan a la misma persona (*hudson*) se parecen entre sí.

**¿Y sin stopwords?** Repito la búsqueda con `stop_words='english'` y `sublinear_tf=True` (que usa 1 + log(tf) y atenúa las palabras repetidas):

In [26]:
vect_sin_stop = TfidfVectorizer(stop_words='english', sublinear_tf=True)
X_train_sin_stop = vect_sin_stop.fit_transform(newsgroups_train.data)

conteos = np.array([[np.sum(y_train[vecinos(i, X)[0]] == y_train[i]) for X in (X_train, X_train_sin_stop)]
                    for i in docs_elegidos])
for i, (con_stop, sin_stop) in zip(docs_elegidos, conteos):
    print(f'{i:>5} · {target_names[y_train[i]]:<24} · misma clase: plantilla {con_stop}/5 · sin stopwords {sin_stop}/5')
print(f'Total: plantilla {conteos[:, 0].sum()}/25 · sin stopwords {conteos[:, 1].sum()}/25')

 8754 · talk.religion.misc       · misma clase: plantilla 4/5 · sin stopwords 4/5
 4965 · comp.sys.mac.hardware    · misma clase: plantilla 2/5 · sin stopwords 2/5
 7404 · comp.os.ms-windows.misc  · misma clase: plantilla 0/5 · sin stopwords 0/5
 1009 · talk.politics.guns       · misma clase: plantilla 4/5 · sin stopwords 3/5
 4899 · sci.crypt                · misma clase: plantilla 3/5 · sin stopwords 0/5
Total: plantilla 13/25 · sin stopwords 9/25


Sin stopwords, los mismos documentos tienen **9/25** vecinos de la misma clase, contra 13/25 con la plantilla. El 4899 pasa de 3 a 0: sus aciertos venían de las stopwords, no del tema, y sin ellas sus vecinos quedan repartidos entre clases. Cinco documentos son muy pocos para comparar representaciones: en el punto 2, medido sobre todo el test, sacar stopwords **sí** mejora la clasificación por similitud.

## 2. Clasificador por prototipos (zero-shot)

Cada documento de test se compara con **todos** los de train y recibe la clase del más parecido. Es un vecino más cercano (1-NN) con similitud coseno: no entrena nada y usa el train tal cual, como "prototipos".

`TfidfVectorizer` deja cada fila con norma L2 igual a 1, así que la similitud coseno es el producto punto: `X_test @ X_train.T`. Lo calculo en bloques de 1.000 documentos para no armar de una la matriz densa de 7.532 × 11.314.

In [27]:
def clasificar_por_prototipos(X_tr, y_tr, X_te, bloque=1000):
    """Asigna a cada fila de X_te la clase del documento de X_tr con mayor similitud coseno."""
    pred = np.empty(X_te.shape[0], dtype=int)
    for inicio in range(0, X_te.shape[0], bloque):
        # Las filas de TF-IDF tienen norma 1: el producto punto es la similitud coseno.
        sim = (X_te[inicio:inicio + bloque] @ X_tr.T).toarray()
        pred[inicio:inicio + bloque] = y_tr[sim.argmax(axis=1)]
    return pred


y_pred_proto = clasificar_por_prototipos(X_train, y_train, X_test)
f1_proto = f1_score(y_test, y_pred_proto, average='macro')

rng = np.random.default_rng(0)
f1_azar = f1_score(y_test, rng.integers(0, 20, size=len(y_test)), average='macro')
f1_baseline = f1_score(y_test, y_pred, average='macro')

print(f'F1 macro en test · azar: {f1_azar:.3f} · prototipos: {f1_proto:.3f} · Naïve Bayes de la plantilla: {f1_baseline:.3f}')
print(f'Accuracy de los prototipos: {np.mean(y_pred_proto == y_test):.3f}')

F1 macro en test · azar: 0.049 · prototipos: 0.505 · Naïve Bayes de la plantilla: 0.585
Accuracy de los prototipos: 0.509


In [28]:
def confusiones_frecuentes(y_real, y_predicho, k=8):
    """Muestra los k pares (clase real → clase predicha) más frecuentes entre los errores."""
    cm = confusion_matrix(y_real, y_predicho)
    np.fill_diagonal(cm, 0)
    for plano in np.argsort(cm, axis=None)[::-1][:k]:
        real, predicha = np.unravel_index(plano, cm.shape)
        print(f'{cm[real, predicha]:>4} · {target_names[real]} → {target_names[predicha]}')


print(classification_report(y_test, y_pred_proto, target_names=target_names, digits=3))
print('Confusiones más frecuentes:')
confusiones_frecuentes(y_test, y_pred_proto)

                          precision    recall  f1-score   support

             alt.atheism      0.366     0.508     0.425       319
           comp.graphics      0.543     0.483     0.512       389
 comp.os.ms-windows.misc      0.506     0.457     0.480       394
comp.sys.ibm.pc.hardware      0.515     0.520     0.518       392
   comp.sys.mac.hardware      0.535     0.499     0.516       385
          comp.windows.x      0.701     0.592     0.642       395
            misc.forsale      0.629     0.462     0.533       390
               rec.autos      0.406     0.576     0.476       396
         rec.motorcycles      0.635     0.515     0.569       398
      rec.sport.baseball      0.645     0.537     0.586       397
        rec.sport.hockey      0.748     0.722     0.735       399
               sci.crypt      0.552     0.588     0.570       396
         sci.electronics      0.531     0.328     0.406       393
                 sci.med      0.652     0.492     0.561       396
         

In [29]:
vacios = X_test.getnnz(axis=1) == 0
print(f'Documentos de test sin ninguna palabra del vocabulario: {vacios.sum()}')
print(f'Clase que les asigna el 1-NN: {set(target_names[c] for c in y_pred_proto[vacios])}')
print(f'Clase del primer documento del train: {target_names[y_train[0]]}')
print(f'F1 macro sin contarlos: {f1_score(y_test[~vacios], y_pred_proto[~vacios], average="macro"):.3f}')

Documentos de test sin ninguna palabra del vocabulario: 224
Clase que les asigna el 1-NN: {'rec.autos'}
Clase del primer documento del train: rec.autos
F1 macro sin contarlos: 0.517


In [30]:
X_test_sin_stop = vect_sin_stop.transform(newsgroups_test.data)
y_pred_proto_sin_stop = clasificar_por_prototipos(X_train_sin_stop, y_train, X_test_sin_stop)
f1_proto_sin_stop = f1_score(y_test, y_pred_proto_sin_stop, average='macro')
print(f'F1 macro de los prototipos sin stopwords y con sublinear_tf: {f1_proto_sin_stop:.3f}')

F1 macro de los prototipos sin stopwords y con sublinear_tf: 0.545


**Análisis**

- **Funciona mucho mejor que el azar:** F1 macro 0.505 contra 0.049, unas diez veces más. Con solo comparar qué palabras comparten, ya se recupera el tema de la mitad de los documentos.
- **Pero queda por debajo del Naïve Bayes de la plantilla** (0.585), por varios motivos:
  1. **Decide un solo vecino.** Basta un documento ruidoso o raro (como los *"Hello,"* del punto 1) para equivocarse. Naïve Bayes, en cambio, junta la evidencia de todos los documentos de la clase.
  2. **Documentos vacíos.** 224 documentos de test no tienen ninguna palabra del vocabulario: quedaron vacíos al sacar encabezados, firmas y citas. Su vector es todo ceros, la similitud con todo el train da 0 y `argmax` devuelve el primer documento, que es de `rec.autos`. Por eso `rec.autos` tiene recall 0.576 pero precisión 0.406. Sin contarlos, el F1 macro sube a 0.517.
  3. **Clases vecinas.** Las confusiones más frecuentes son entre grupos del mismo tema: `soc.religion.christian` → `alt.atheism`, `talk.politics.misc` → `talk.politics.guns`, `comp.windows.x` → `comp.graphics`, Mac → PC. Comparten vocabulario, y la bolsa de palabras no capta los matices. Las peores clases son las *misc* (`talk.religion.misc` 0.277, `talk.politics.misc` 0.307), que mezclan temas de sus grupos hermanos. La mejor es `rec.sport.hockey` (0.735), con vocabulario muy propio (equipos, NHL).
  4. **Las stopwords pesan en el coseno.** En los textos largos, las palabras comunes inflan la similitud (punto 1). Sin stopwords y con `sublinear_tf`, el mismo clasificador sube de 0.505 a 0.545.
- **Costo:** no hay entrenamiento, pero cada predicción compara el documento con los 11.314 del train. Todo el trabajo se pasa al momento de predecir.

## 3. Naïve Bayes

**Método.** El test no se usa para elegir nada: cada configuración se evalúa con **validación cruzada estratificada de 5 folds sobre el train** (F1 macro), y el test se mira una sola vez al final, con las configuraciones elegidas. Si se eligieran los parámetros mirando el test, el F1 reportado sería optimista.

`cv_f1` vectoriza cada fold una sola vez (ajustando el vectorizador solo con la parte de entrenamiento del fold) y prueba varios modelos sobre esa vectorización. `ngram_range` queda siempre en `(1, 1)`, como pide la consigna.

In [31]:
docs_train = np.array(newsgroups_train.data, dtype=object)
folds = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=0).split(docs_train, y_train))


def cv_f1(params_vectorizador, modelos):
    """F1 macro promedio en los 5 folds de cada modelo, con TfidfVectorizer(**params_vectorizador)."""
    scores = {nombre: [] for nombre in modelos}
    for idx_tr, idx_val in folds:
        vect = TfidfVectorizer(**params_vectorizador)
        X_tr = vect.fit_transform(docs_train[idx_tr])
        X_val = vect.transform(docs_train[idx_val])
        for nombre, modelo in modelos.items():
            pred = clone(modelo).fit(X_tr, y_train[idx_tr]).predict(X_val)
            scores[nombre].append(f1_score(y_train[idx_val], pred, average='macro'))
    return {nombre: np.mean(s) for nombre, s in scores.items()}

### 3.1 El suavizado: `alpha`

Con el vectorizador de la plantilla, pruebo distintos `alpha` en los dos modelos.

In [32]:
alphas = [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0]
modelos = {**{('MultinomialNB', a): MultinomialNB(alpha=a) for a in alphas},
           **{('ComplementNB', a): ComplementNB(alpha=a) for a in alphas}}
res_alpha = pd.Series(cv_f1({}, modelos)).unstack(level=0)
res_alpha.index.name = 'alpha'
res_alpha.round(3)

,ComplementNB,MultinomialNB
alpha,,
0.001,0.714,0.748
0.003,0.724,0.755
0.010,0.739,0.758
0.030,0.749,0.746
0.100,0.762,0.718
0.300,0.762,0.683
1.000,0.752,0.625


Para entender el efecto de `alpha`, comparo el peso TF-IDF que junta cada clase con lo que suma el suavizado, y cuántas veces predice cada clase el baseline (`alpha=1`):

In [33]:
peso = pd.DataFrame({
    'docs train': np.bincount(y_train),
    'peso TF-IDF': [X_train[y_train == c].sum() for c in range(20)],
    'docs test': np.bincount(y_test),
    'predichos por el baseline': np.bincount(y_pred, minlength=20),
}, index=target_names).sort_values('peso TF-IDF')

print(f'Suma de los valores TF-IDF de un documento (promedio): {X_train.sum(axis=1).mean():.1f}')
print(f'Peso TF-IDF de una clase (promedio): {peso["peso TF-IDF"].mean():,.0f}')
print(f'Suavizado total por clase · alpha=1: {1.0 * X_train.shape[1]:,.0f} · alpha=0.01: {0.01 * X_train.shape[1]:,.0f}')
print(f'Términos que aparecen en un solo documento: {(X_train.getnnz(axis=0) == 1).sum():,}')
peso.round(0)

Suma de los valores TF-IDF de un documento (promedio): 7.0
Peso TF-IDF de una clase (promedio): 3,944
Suavizado total por clase · alpha=1: 101,631 · alpha=0.01: 1,016
Términos que aparecen en un solo documento: 62,208


,docs train,peso TF-IDF,docs test,predichos por el baseline
talk.religion.misc,377,2729.0,251,2
alt.atheism,480,3376.0,319,27
comp.sys.mac.hardware,578,3564.0,385,287
talk.politics.misc,465,3589.0,310,31
comp.graphics,584,3609.0,389,334
rec.sport.baseball,597,3729.0,397,290
comp.os.ms-windows.misc,591,3739.0,394,282
misc.forsale,585,3746.0,390,313
rec.motorcycles,598,3750.0,398,280
rec.autos,594,3762.0,396,329


**Análisis**

- **MultinomialNB depende muchísimo de `alpha`:** pasa de 0.625 con `alpha=1` (el valor por defecto, el del baseline) a 0.758 con `alpha=0.01`.
- **Por qué:** `alpha` es el suavizado de Laplace, y se suma a cada uno de los 101.631 términos en cada clase. Con TF-IDF cada documento aporta poco peso (sus valores suman ≈ 7, por la normalización L2), así que cada clase junta ≈ 3.900 de peso real. Con `alpha=1` el suavizado agrega 101.631, unas 26 veces más que la evidencia: las probabilidades P(palabra | clase) quedan casi uniformes y el modelo apenas distingue las clases. Con `alpha=0.01`, el suavizado (≈ 1.000) queda por debajo de la evidencia.
- **Con el suavizado excesivo ganan las clases con más peso.** El baseline casi nunca predice `talk.religion.misc`, la clase con menos peso TF-IDF (2 veces en 7.532 documentos, cuando hay 251), y vuelca todo en `soc.religion.christian`, la de más peso (1.710 predicciones para 398 documentos reales). Entre clases de vocabulario parecido (religión, ateísmo), gana la que más palabras vio.
- **Con `alpha` demasiado chico vuelve a bajar** (0.001): el modelo confía demasiado en palabras que vio una o dos veces, y 62.208 de los términos aparecen en un solo documento.
- **ComplementNB es más estable:** estima los parámetros de cada clase con los documentos de las otras 19 (su complemento), que son muchos más, así que el suavizado pesa menos y las clases chicas no quedan en desventaja. Su mejor `alpha` es más alto (0.1–0.3), y con `alpha=1` todavía da 0.752.

### 3.2 El vectorizador

Cambio un parámetro de `TfidfVectorizer` por vez. Como el mejor `alpha` depende de la escala de los valores, para cada variante lo vuelvo a elegir entre 0.01 y 1. Las dos últimas variantes no usan IDF ni normalización: equivalen a un `CountVectorizer` (conteos) y a una bolsa de palabras binaria, como las de la clase.

In [34]:
variantes = {
    'plantilla': {},
    "stop_words='english'": dict(stop_words='english'),
    'sublinear_tf=True': dict(sublinear_tf=True),
    'stop_words + sublinear_tf': dict(stop_words='english', sublinear_tf=True),
    'max_df=0.5': dict(max_df=0.5),
    'min_df=2': dict(min_df=2),
    'min_df=5': dict(min_df=5),
    'solo letras (sin números)': dict(token_pattern=r'(?u)\b[^\W\d_]{2,}\b'),
    'conteos (sin IDF ni norma)': dict(use_idf=False, norm=None),
    'binario (sin IDF ni norma)': dict(binary=True, use_idf=False, norm=None),
}
alphas_cortos = [0.01, 0.03, 0.1, 0.3, 1.0]
modelos = {**{('MultinomialNB', a): MultinomialNB(alpha=a) for a in alphas_cortos},
           **{('ComplementNB', a): ComplementNB(alpha=a) for a in alphas_cortos}}

filas = []
for nombre, params in variantes.items():
    res = pd.Series(cv_f1(params, modelos))
    fila = {'vectorizador': nombre,
            'términos': len(TfidfVectorizer(**params).fit(newsgroups_train.data).vocabulary_)}
    for modelo in ['MultinomialNB', 'ComplementNB']:
        fila[f'{modelo} F1 (CV)'] = res[modelo].max()
        fila[f'{modelo} alpha'] = res[modelo].idxmax()
    filas.append(fila)
res_vect = pd.DataFrame(filas).set_index('vectorizador')
res_vect.round(3)

,términos,MultinomialNB F1 (CV),MultinomialNB alpha,ComplementNB F1 (CV),ComplementNB alpha
vectorizador,,,,,
plantilla,101631,0.758,0.01,0.762,0.1
stop_words='english',101322,0.760,0.01,0.762,0.3
sublinear_tf=True,101631,0.753,0.01,0.763,0.3
stop_words + sublinear_tf,101322,0.754,0.01,0.764,0.3
max_df=0.5,101622,0.759,0.01,0.762,0.3
min_df=2,39423,0.752,0.01,0.754,0.3
min_df=5,18101,0.737,0.03,0.731,1.0
solo letras (sin números),67741,0.754,0.01,0.756,0.3
conteos (sin IDF ni norma),101631,0.689,0.03,0.710,0.3


**Análisis**

- **Las variantes de TF-IDF que no filtran por frecuencia dan casi lo mismo** (entre 0.753 y 0.764). Es una diferencia chica comparada con el efecto de `alpha`, que en MultinomialNB supera 0.13.
- **`stop_words='english'`** mejora apenas (de 0.758 a 0.760 en MultinomialNB). El IDF ya les baja el peso a las palabras que aparecen en casi todos los documentos: como se dijo en clase, es otra forma de limpiar stopwords sin diccionario. Además, las stopwords aparecen parecido en todas las clases, así que no inclinan la decisión de Naïve Bayes.
- **`max_df=0.5`** solo saca 9 términos (*the*, *to*, *of*, *and*, *in*, *is*, *it*, *that*, *for*), que el IDF ya casi anulaba: el resultado prácticamente no cambia.
- **`sublinear_tf=True`** atenúa las palabras repetidas en un mismo documento: ayuda un poco a ComplementNB y empeora un poco a MultinomialNB.
- **`min_df` empeora:** reduce el vocabulario de 101.631 a 39.423 términos (`min_df=2`) y a 18.101 (`min_df=5`), pero las palabras raras (nombres, modelos de productos, jerga) son muy indicativas de una clase. La ley de Zipf decía que las muy raras no sirven para medir **similitud**; para **clasificar**, una palabra que aparece en tres documentos de la misma clase es una buena pista.
- **Solo letras** también empeora un poco: los números informan (modelos de procesadores como *486*, años, números de versión).
- **Sin IDF ni normalización cae mucho** (0.689 en MultinomialNB, 0.710 en ComplementNB): los documentos largos dominan los conteos de su clase y las palabras comunes pesan tanto como las específicas. La versión **binaria** queda en el medio: importa más si una palabra aparece que cuántas veces.

Con validación cruzada, la mejor configuración de MultinomialNB es `stop_words='english'` con `alpha=0.01` (0.760), y la de ComplementNB, `stop_words` + `sublinear_tf` con `alpha=0.3` (0.764).

### 3.3 Los modelos elegidos, en test

Para cada modelo tomo la variante y el `alpha` con mejor F1 en validación cruzada, lo entreno con todo el train y lo evalúo una sola vez en test.

In [35]:
f1_test, pred_test = {}, {}
for modelo, clase_nb in [('MultinomialNB', MultinomialNB), ('ComplementNB', ComplementNB)]:
    variante = res_vect[f'{modelo} F1 (CV)'].idxmax()
    alpha = res_vect.loc[variante, f'{modelo} alpha']
    vect = TfidfVectorizer(**variantes[variante])
    nb = clase_nb(alpha=alpha).fit(vect.fit_transform(newsgroups_train.data), y_train)
    pred_test[modelo] = nb.predict(vect.transform(newsgroups_test.data))
    f1_test[modelo] = f1_score(y_test, pred_test[modelo], average='macro')
    print(f'{modelo} · {variante} · alpha={alpha} · F1 macro CV {res_vect.loc[variante, f"{modelo} F1 (CV)"]:.3f}'
          f' · test {f1_test[modelo]:.3f} · accuracy test {np.mean(pred_test[modelo] == y_test):.3f}')
print(f'Baseline de la plantilla (MultinomialNB, alpha=1, TF-IDF por defecto) · test {f1_baseline:.3f}')

MultinomialNB · stop_words='english' · alpha=0.01 · F1 macro CV 0.760 · test 0.684 · accuracy test 0.701


ComplementNB · stop_words + sublinear_tf · alpha=0.3 · F1 macro CV 0.764 · test 0.698 · accuracy test 0.717
Baseline de la plantilla (MultinomialNB, alpha=1, TF-IDF por defecto) · test 0.585


In [36]:
mejor_modelo = max(f1_test, key=f1_test.get)
por_clase = pd.DataFrame({
    'prototipos (1-NN)': f1_score(y_test, y_pred_proto, average=None),
    'NB plantilla': f1_score(y_test, y_pred, average=None),
    f'{mejor_modelo} elegido': f1_score(y_test, pred_test[mejor_modelo], average=None),
}, index=target_names)
por_clase.sort_values(f'{mejor_modelo} elegido').round(3)

,prototipos (1-NN),NB plantilla,ComplementNB elegido
talk.religion.misc,0.277,0.008,0.224
alt.atheism,0.425,0.127,0.369
talk.politics.misc,0.307,0.147,0.524
sci.electronics,0.406,0.549,0.632
comp.os.ms-windows.misc,0.480,0.586,0.635
talk.politics.guns,0.462,0.588,0.655
comp.sys.ibm.pc.hardware,0.518,0.638,0.670
soc.religion.christian,0.508,0.358,0.695
comp.graphics,0.512,0.667,0.725
misc.forsale,0.533,0.768,0.739


In [37]:
print(f'Confusiones más frecuentes de {mejor_modelo}:')
confusiones_frecuentes(y_test, pred_test[mejor_modelo])

Confusiones más frecuentes de ComplementNB:
 106 · talk.religion.misc → soc.religion.christian
  91 · alt.atheism → soc.religion.christian
  88 · talk.politics.misc → talk.politics.guns
  50 · comp.os.ms-windows.misc → comp.sys.ibm.pc.hardware
  35 · comp.windows.x → comp.graphics
  33 · talk.religion.misc → alt.atheism
  33 · sci.electronics → sci.crypt
  31 · misc.forsale → comp.sys.ibm.pc.hardware


**Análisis**

- **En test, ComplementNB llega a 0.698 y MultinomialNB a 0.684**, contra 0.585 del baseline: +0.11 solo eligiendo bien `alpha` y el vectorizador.
- **La validación cruzada es optimista** (≈ 0.76 contra ≈ 0.70 en test). 20 Newsgroups viene dividido por fecha: los mensajes de test son posteriores y traen hilos y temas nuevos. Dentro del train, en cambio, los folds comparten hilos, citas y autores (como *hudson* en el punto 1), y validar ahí es más fácil.
- **Las clases que más mejoran son las que el baseline casi no predecía:** `talk.politics.misc` (0.147 → 0.524), `soc.religion.christian` (0.358 → 0.695), `alt.atheism` (0.127 → 0.369) y `talk.religion.misc` (0.008 → 0.224). Con menos suavizado, y con ComplementNB, las clases chicas dejan de estar en desventaja.
- **Las que siguen peor son las mismas que confundía el 1-NN:** religión (`talk.religion.misc` y `alt.atheism` → `soc.religion.christian`), política (`talk.politics.misc` → `talk.politics.guns`) y hardware (`comp.os.ms-windows.misc` → `comp.sys.ibm.pc.hardware`). Son grupos con vocabulario compartido, donde lo que cambia es el enfoque o la opinión, y eso una bolsa de palabras no lo ve.
- **Lo mejor clasificado son los deportes** (`rec.sport.hockey` 0.905, `rec.sport.baseball` 0.882), que tienen vocabulario muy propio.

## 4. Similitud entre palabras

Al transponer la matriz documento-término, cada **palabra** queda representada por un vector de 11.314 dimensiones: su peso TF-IDF en cada documento del train. Dos palabras se parecen si aparecen en los mismos documentos.

Elijo cinco palabras de temas distintos: *gun* (armas), *israel* (Medio Oriente), *drive* (hardware), *key* (criptografía, aunque tiene otros sentidos) y *god* (religión). Junto a cada vecino muestro su similitud y su `df`, la cantidad de documentos en que aparece.

In [38]:
def palabras_similares(palabra, vect=tfidfvect, X_docs=X_train, k=5):
    """Las k palabras más parecidas a `palabra` según la matriz término-documento (X_docs transpuesta)."""
    X_t = X_docs.T.tocsr()
    voc = vect.get_feature_names_out()
    df = X_t.getnnz(axis=1)  # en cuántos documentos aparece cada término
    i = vect.vocabulary_[palabra]
    sim = cosine_similarity(X_t[i], X_t).ravel()
    sim[i] = -1
    top = np.argsort(sim)[::-1][:k]
    return f'{palabra} (df={df[i]}): ' + ', '.join(f'{voc[j]} {sim[j]:.2f} (df={df[j]})' for j in top)


print(f'Matriz término-documento: {X_train.shape[1]:,} términos × {X_train.shape[0]:,} documentos\n')
for palabra in ['gun', 'israel', 'drive', 'key', 'god']:
    print(palabras_similares(palabra))

Matriz término-documento: 101,631 términos × 11,314 documentos

gun (df=235): guns 0.36 (df=156), crime 0.24 (df=172), handgun 0.24 (df=41), homicides 0.23 (df=15), firearms 0.23 (df=76)
israel (df=184): israeli 0.42 (df=130), arab 0.37 (df=95), syrian 0.37 (df=16), lebanon 0.36 (df=48), lebanese 0.34 (df=30)
drive (df=490): drives 0.32 (df=158), hard 0.31 (df=576), floppy 0.31 (df=126), disk 0.25 (df=275), ide 0.25 (df=102)
key (df=384): keys 0.38 (df=185), hellman 0.33 (df=24), escrow 0.32 (df=85), diffie 0.29 (df=16), session 0.27 (df=82)


god (df=579): jesus 0.27 (df=263), bible 0.26 (df=243), that 0.26 (df=6580), existence 0.25 (df=137), christ 0.25 (df=197)

**Análisis**

- **Los vecinos tienen sentido, pero por tema y no por sinonimia:** *gun* → *guns*, *crime*, *handgun*, *homicides*, *firearms*; *israel* → *israeli*, *arab*, *syrian*, *lebanon*, *lebanese*. *crime* no es sinónimo de *gun*: aparecen juntas porque se usan en las mismas discusiones. Esta representación captura **co-ocurrencia en documentos**.
- **Una palabra, un solo sentido:** los vecinos de *drive* son *drives*, *hard*, *floppy*, *disk* e *ide*, y no hay nada sobre manejar un auto. Los de *key* son *keys*, *hellman*, *escrow*, *diffie* y *session*, todo de criptografía (el intercambio de claves Diffie-Hellman, el *key escrow* del chip Clipper), y nada sobre llaves ni teclados. Cada palabra tiene un solo vector, dominado por su uso más frecuente en el corpus: es el problema de "banco" que se vio en clase.
- **Sin stemming ni lematización:** *gun* y *guns*, *key* y *keys* o *drive* y *drives* son vectores distintos, aunque quedan cerca porque aparecen en los mismos documentos.
- **Se cuela una stopword:** *that* (que aparece en 6.580 documentos) es vecina de *god*, porque *god* aparece en textos religiosos largos y argumentativos donde *that* abunda.
- Las similitudes son bajas (0.2–0.4): los vectores son muy esparsos y dos palabras comparten pocos documentos.

**¿Dónde falla?** Con otras palabras igual de razonables, como *encryption* o *apple*, los vecinos son términos que casi nadie usa. Abajo, cada palabra con la matriz de la plantilla y con un vectorizador que saca las stopwords y los términos que aparecen en menos de 10 documentos (`min_df=10`).

In [39]:
vect_filtrado = TfidfVectorizer(stop_words='english', min_df=10)
X_filtrado = vect_filtrado.fit_transform(newsgroups_train.data)
print(f'Vocabulario: {X_train.shape[1]:,} términos en la plantilla → {X_filtrado.shape[1]:,} sin stopwords y con min_df=10\n')

for palabra in ['encryption', 'apple', 'gun', 'israel', 'drive', 'key', 'god']:
    print('plantilla:', palabras_similares(palabra))
    print('filtrado: ', palabras_similares(palabra, vect_filtrado, X_filtrado), '\n')

Vocabulario: 101,631 términos en la plantilla → 10,441 sin stopwords y con min_df=10

plantilla: encryption (df=169): microcircuit 0.34 (df=7), accommodates 0.34 (df=5), torrance 0.34 (df=5), 2758 0.34 (df=5), nistnews 0.34 (df=5)
filtrado:  encryption (df=169): privacy 0.33 (df=90), clipper 0.32 (df=172), encrypted 0.28 (df=84), briefed 0.25 (df=12), unbreakable 0.25 (df=18) 

plantilla: apple (df=191): bcga65431 0.22 (df=1), undisclosed 0.22 (df=1), _macweek_ 0.22 (df=1), 5411 0.22 (df=1), f83y 0.22 (df=1)
filtrado:  apple (df=191): duo 0.20 (df=30), laserwriter 0.18 (df=16), mac 0.14 (df=261), quicktime 0.14 (df=15), adb 0.13 (df=17) 

plantilla: gun (df=235): guns 0.36 (df=156), crime 0.24 (df=172), handgun 0.24 (df=41), homicides 0.23 (df=15), firearms 0.23 (df=76)
filtrado:  gun (df=235): guns 0.36 (df=156), handgun 0.24 (df=41), crime 0.23 (df=172), homicides 0.23 (df=15), firearms 0.23 (df=76) 

plantilla: israel (df=184): israeli 0.42 (df=130), arab 0.37 (df=95), syrian 0.37 (

plantilla: drive (df=490): drives 0.32 (df=158), hard 0.31 (df=576), floppy 0.31 (df=126), disk 0.25 (df=275), ide 0.25 (df=102)
filtrado:  drive (df=490): drives 0.32 (df=158), floppy 0.29 (df=126), hard 0.28 (df=576), disk 0.24 (df=275), ide 0.23 (df=102) 

plantilla: key (df=384): keys 0.38 (df=185), hellman 0.33 (df=24), escrow 0.32 (df=85), diffie 0.29 (df=16), session 0.27 (df=82)
filtrado:  key (df=384): keys 0.39 (df=185), escrow 0.31 (df=85), hellman 0.31 (df=24), diffie 0.27 (df=16), chip 0.26 (df=296) 

plantilla: god (df=579): jesus 0.27 (df=263), bible 0.26 (df=243), that 0.26 (df=6580), existence 0.25 (df=137), christ 0.25 (df=197)
filtrado:  god (df=579): jesus 0.27 (df=263), christ 0.27 (df=197), bible 0.26 (df=243), faith 0.25 (df=191), existence 0.24 (df=137) 



**Análisis**

- **Con la matriz de la plantilla, *encryption* y *apple* tienen vecinos sin sentido.** Los de *encryption* (*microcircuit*, *accommodates*, *torrance*, *2758*, *nistnews*) aparecen en 5 a 7 documentos: salen del comunicado oficial que anunció el chip Clipper, publicado cinco veces en `sci.crypt`. Los de *apple* (*bcga65431*, *undisclosed*, *_macweek_*…) aparecen en un solo documento cada uno.
- **Es la ley de Zipf:** una palabra que aparece en muy pocos documentos tiene su vector concentrado ahí. Si la palabra consultada pesa mucho en esos mismos documentos, el coseno da alto aunque no haya relación de tema. Son las palabras *too rare* de la slide de Zipf.
- **Al filtrar mejora mucho** (de 101.631 a 10.441 términos): *encryption* → *privacy*, *clipper*, *encrypted*, *briefed*, *unbreakable*; *apple* → *duo*, *laserwriter*, *mac*, *quicktime*, *adb* (productos de Apple de la época). *god* cambia *that* por *faith*. Las otras palabras casi no cambian.

**Conclusión del punto 4:** la matriz término-documento ya da una primera representación vectorial de las palabras, basada en los documentos en que aparecen. Sirve para encontrar palabras del mismo tema, pero tiene las limitaciones de la bolsa de palabras: vectores enormes y esparsos (una dimensión por documento), sensibles a las palabras raras, sin sinónimos ni morfología, y con un solo sentido por palabra. Es el punto de partida de las matrices de co-ocurrencia y los embeddings de la clase 2.

## Conclusiones

In [40]:
pd.Series({
    'Azar': f1_azar,
    'Prototipos 1-NN (TF-IDF de la plantilla)': f1_proto,
    'Prototipos 1-NN (sin stopwords, sublinear_tf)': f1_proto_sin_stop,
    'MultinomialNB alpha=1 (baseline)': f1_baseline,
    'MultinomialNB elegido por CV': f1_test['MultinomialNB'],
    'ComplementNB elegido por CV': f1_test['ComplementNB'],
}, name='F1 macro en test').round(3).to_frame()

,F1 macro en test
Azar,0.049
Prototipos 1-NN (TF-IDF de la plantilla),0.505
"Prototipos 1-NN (sin stopwords, sublinear_tf)",0.545
MultinomialNB alpha=1 (baseline),0.585
MultinomialNB elegido por CV,0.684
ComplementNB elegido por CV,0.698


1. **La bolsa de palabras con TF-IDF alcanza para reconocer el tema.** Comparar documentos por coseno ya es diez veces mejor que el azar, y un Naïve Bayes bien configurado clasifica bien a 7 de cada 10 documentos (F1 macro ≈ 0.70).
2. **En Naïve Bayes, lo que más importa es `alpha`.** Su valor por defecto (1) es enorme para la escala de TF-IDF: aplana las probabilidades y hace que el modelo casi no prediga las clases chicas. Bajarlo lleva el F1 de 0.585 a 0.684, y ComplementNB, menos sensible al suavizado, llega a 0.698.
3. **El IDF ya hace casi todo el trabajo de las stopwords** en Naïve Bayes. En la similitud coseno, en cambio, sacarlas sí ayuda: el 1-NN sube de 0.505 a 0.545.
4. **Los errores se concentran en grupos de temas cercanos** (religión, política, hardware de PC y Mac). Ahí cambian el enfoque y la opinión más que el vocabulario, y una representación sin orden ni significado no alcanza.
5. **La matriz término-documento da vectores de palabras por tema,** pero son esparsos, sensibles a las palabras raras y no distinguen sinónimos ni sentidos. Es lo que vienen a resolver los embeddings de la clase 2.